# house

> The rules a periodic register must not be scored against

In [ ]:
#| default_exp house

The meter's implicit ideal is the short declarative sentence. That ideal is not universal, and it is not this fork's: a house style built on the periodic sentence is scored by the stock rules as though its virtues were faults.

Measured, on the reference text of the register in question. Samuel Johnson's *Rambler* No. 60 scores **28.2** under the stock rules, and the heaviest single finding, at weight 16, is his opening period, for length. Two more fall on `very` and `few`, which are ordinary English rather than hedging, and two on his passives. Under this profile the same passage scores **0.0**.

What the profile does not touch is the larger half of the meter, and the half worth having: banned vocabulary, not-X-but-Y, throat-clearing, noting fillers, transition glue, the rule of three, moulded bullet lists, heading echo, coinage, nominalization. A paragraph of machine slop scores 279.5 under the stock rules and 253.8 under this one. Those rules detect cant, and cant is the enemy of a good style in any century.

In [ ]:
#| export
import os

from fastcore.utils import *

from slopometer import lexicon
from slopometer.core import rules, SMELL
from slopometer.lexicon import lex_rule, hedges

## What the profile drops

Seven rules, each of which fires on the register rather than on a fault.

In [ ]:
#| export
HOUSE_DROP = {
    'sent_len':    'the periodic sentence is the form, not a defect',
    'clauses':     'subordination is how a period is built',
    'semi_splice': "the semicolon join is the register's chief instrument",
    'splice':      'so is the dash aside',
    'passive':     '"What is done from necessity..." is Johnson\'s sentence, not a lapse',
    'variation':   'two words for one thing is a deliberate device here, not an accident',
    'conseq':      '", so" as consequence glue reads as speech, which is wanted',
}

## What the profile keeps, in amended form

`hedges` bundles genuine hedging with the weasel-word list adopted from write-good. The genuine hedges stay. The ordinary English words go, because `very`, `few` and `several` are not evasions, and the `intensifiers` rule already catches the intensifier misuse that motivated them.

In [ ]:
#| export
HOUSE_HEDGES = {k: v for k, v in hedges.items() if k in {
    'potentially', 'in some cases', 'should generally', 'may or may not',
    'one might argue', 'it could be argued', 'arguably', 'more or less'}}

## Applying it

Two details of the surrounding machinery shape this function.

The rules register themselves when their module is imported, and `house` imports only `core` and `lexicon`, so five of the seven are not yet in the registry when this module loads. The function therefore imports `syntax` and `para` before it prunes anything.

And the command line runs through `warmpy`, whose worker outlives the call: a later invocation reuses a process started under whichever profile came first, so its environment is read once and no more. The switch is therefore reversible. Dropped rules are stashed and restored, rather than lost, which makes `SLOPOMETER_PROFILE=upstream` work on a warm worker as well as a cold one.

In [ ]:
#| export
_stash = {}


def apply_profile(
    name: str = None,  # Profile to apply; `$SLOPOMETER_PROFILE` when None, and 'house' when that is unset
) -> list:
    "Amend the rule registry in place for `name`, returning the rules this call removed"
    from slopometer import syntax, para  # every rule must be registered before any is pruned
    if (name or os.environ.get('SLOPOMETER_PROFILE', 'house')) != 'house':
        rules.update(_stash)
        _stash.clear()
        lexicon.find_hedges = lex_rule('hedges', tell=7, weight=SMELL, lex=hedges)
        return []
    out = []
    for n in HOUSE_DROP:
        if (r := rules.pop(n, None)) is not None:
            _stash[n] = r
            out.append(n)
    lexicon.find_hedges = lex_rule('hedges', tell=7, weight=SMELL, lex=HOUSE_HEDGES)
    return out

The command line applies this on every run, which is what the Claude Code slop hook shells out to. A library caller asks for it explicitly, before scoring:

```python
from slopometer.house import apply_profile
apply_profile()
```

In [ ]:
dropped = apply_profile()
assert set(dropped) == set(HOUSE_DROP), dropped
assert not (set(HOUSE_DROP) & set(rules)), 'a dropped rule is still registered'
assert 'hedges' in rules, 'hedges is amended, not dropped'
assert apply_profile() == [], 'applying twice removes nothing the second time'
dropped

In [ ]:
assert rules['hedges']('this is very important') == [], 'ordinary English is no longer hedging'
assert lexicon.find_hedges('this is very important') == [], 'the module-level name is rebound too, not left stale'
assert len(rules['hedges']('it could be argued, potentially')) == 2, 'real hedges still fire'
rules['hedges']('it could be argued, potentially')

In [ ]:
apply_profile('upstream')
assert set(HOUSE_DROP) <= set(rules), 'upstream restores every dropped rule'
assert lexicon.find_hedges('this is very important'), 'upstream restores the weasel list'
apply_profile()  # leave the house profile in force
sorted(rules)

In [ ]:
from slopometer.score import score_text

johnson = ('It very seldom happens to man that his business is his pleasure. What is done from necessity '
           'is so often to be done when against the present inclination, and so often fills the mind with '
           'anxiety, that an habitual dislike steals upon us, and we shrink involuntarily from the '
           'remembrance of our task.')
slop = ("This section describes our comprehensive approach. It isn't just a linter - it's a robust "
        'paradigm for quality. Furthermore, it is worth noting that we leverage cutting-edge techniques.')

assert score_text(johnson).density == 0.0, score_text(johnson)
assert score_text(slop).density > 200, score_text(slop)
score_text(slop)